# RQ6: Discount Optimization — Impact of Discount Level on Campaign Performance

**Research Question:** How does discount level affect campaign ROI and customer satisfaction, and what is the optimal discount range for each subscription tier that maximizes ROI while maintaining high satisfaction?

**Task:** Regression Analysis + Polynomial Optimization  
**Targets:** ROI, Customer_Satisfaction_Post_Refund  
**Predictor:** Discount_Level (10–70%)  
**Methods:** Polynomial Regression (degree 2 & 3), Binned Analysis, Segmented by Subscription_Tier  
**Dataset:** Marketing and Product Performance Dataset (Kaggle)

In [89]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3', '#FF5722', '#4CAF50']
TIER_PALETTE = {'Basic': '#2196F3', 'Standard': '#FF5722', 'Premium': '#4CAF50'}
RANDOM_STATE = 42
OUTPUT_DIR = '/kaggle/working/'

def save_figure(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.savefig(path, format='pdf', bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f'Saved figure: {path}')

def save_table(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f'Saved table:  {path}')

def fit_poly(x, y, degree=2):
    pipe = Pipeline([('poly', PolynomialFeatures(degree=degree, include_bias=True)),
                     ('lr',   LinearRegression())])
    pipe.fit(x.reshape(-1, 1), y)
    return pipe

print('Imports OK')

Imports OK


## 1. Data Loading

In [90]:
input_dir = '/kaggle/input'
data_files = [
    os.path.join(root, f)
    for root, dirs, files in os.walk(input_dir)
    for f in files if f.endswith('.xlsx') or f.endswith('.xls') or f.endswith('.csv')
]
print('Found files:', data_files)
FILE_PATH = data_files[0]

df = pd.read_csv(FILE_PATH) if FILE_PATH.endswith('.csv') else pd.read_excel(FILE_PATH)
print(f'Shape: {df.shape}')

TIER_COL = 'Subscription_Tier'
TIERS    = sorted(df[TIER_COL].dropna().unique())
print(f'Discount_Level range: {df["Discount_Level"].min()} – {df["Discount_Level"].max()}')
df.head()

Found files: ['/kaggle/input/datasets/vanishjr/marketing-product-performance/marketing_and_product_performance.csv']
Shape: (10000, 17)
Discount_Level range: 10 – 69


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative
2,CMP_6SBOWN,PROD_4V8A08,22367.45,3546,265,32585.62,1.41,CUST_BR2GST,Basic,9,FLASH_J4PEON,51,160,BNDL_0HY0EF,462.87,4,Affordable
3,CMP_Q31QCU,PROD_A1Q6ZB,29957.54,2573,781,95740.12,3.32,CUST_6TBY6K,Premium,32,FLASH_1TOVXT,36,159,BNDL_AI09BC,334.16,1,Durable
4,CMP_AY0UTJ,PROD_F57N66,36277.19,818,79,81990.43,3.53,CUST_XASI45,Standard,29,FLASH_AOBHXL,20,52,BNDL_R03ITT,371.67,2,Affordable


## 2. EDA — Discount Level vs ROI & Satisfaction

In [91]:
# Pearson correlations
r_roi,  p_roi  = pearsonr(df['Discount_Level'].dropna(), df['ROI'].dropna())
r_sat,  p_sat  = pearsonr(df['Discount_Level'].dropna(),
                           df['Customer_Satisfaction_Post_Refund'].dropna())
r_rev,  p_rev  = pearsonr(df['Discount_Level'].dropna(), df['Revenue_Generated'].dropna())
print(f'Discount_Level vs ROI:          r={r_roi:.4f}  p={p_roi:.4f}')
print(f'Discount_Level vs Satisfaction: r={r_sat:.4f}  p={p_sat:.4f}')
print(f'Discount_Level vs Revenue:      r={r_rev:.4f}  p={p_rev:.4f}')

Discount_Level vs ROI:          r=0.0099  p=0.3224
Discount_Level vs Satisfaction: r=-0.0141  p=0.1579
Discount_Level vs Revenue:      r=0.0024  p=0.8111


In [92]:
# Binned summary table
bins   = list(range(10, 75, 10))  # [10,20,30,40,50,60,70]
labels = [f'{b}-{b+10}%' for b in bins[:-1]]
df['Discount_Bin'] = pd.cut(df['Discount_Level'], bins=bins, labels=labels, right=False, include_lowest=True)

bin_summary = df.groupby('Discount_Bin', observed=True).agg(
    Mean_ROI=('ROI', 'mean'),
    Mean_Revenue=('Revenue_Generated', 'mean'),
    Mean_Satisfaction=('Customer_Satisfaction_Post_Refund', 'mean'),
    Mean_Units_Sold=('Units_Sold', 'mean'),
    Count=('ROI', 'count')
).round(4).reset_index()

save_table(bin_summary, 'rq6_discount_performance_summary.csv')
bin_summary

Saved table:  /kaggle/working/rq6_discount_performance_summary.csv


,Discount_Bin,Mean_ROI,Mean_Revenue,Mean_Satisfaction,Mean_Units_Sold,Count
0,10-20%,2.7087,50216.4268,2.5061,100.1250,1648
1,20-30%,2.7375,50634.7921,2.5140,101.2347,1683
2,30-40%,2.7932,48732.3728,2.4915,100.3437,1705
3,40-50%,2.7974,50306.3687,2.5622,100.7078,1663
4,50-60%,2.7288,50471.2377,2.4782,99.7899,1675
5,60-70%,2.7720,49891.5986,2.4526,101.9686,1626


## 3. Polynomial Regression — ROI vs Discount Level by Tier

In [93]:
x_smooth = np.linspace(df['Discount_Level'].min(), df['Discount_Level'].max(), 300)
DEGREE   = 3

fig, ax = plt.subplots(figsize=(11, 6))

optimal_rows = []
for tier, color in TIER_PALETTE.items():
    sub = df[df[TIER_COL] == tier].dropna(subset=['Discount_Level', 'ROI'])
    if len(sub) < 10:
        continue
    x = sub['Discount_Level'].values
    y = sub['ROI'].values

    # Scatter
    ax.scatter(x, y, alpha=0.15, s=15, color=color, edgecolors='none')

    # Polynomial fit
    poly_pipe = fit_poly(x, y, degree=DEGREE)
    y_smooth  = poly_pipe.predict(x_smooth.reshape(-1, 1))
    r2        = r2_score(y, poly_pipe.predict(x.reshape(-1, 1)))

    ax.plot(x_smooth, y_smooth, color=color, linewidth=2.5,
            label=f'{tier} (R²={r2:.3f})')

    # Optimal discount (argmax of polynomial curve)
    opt_discount = round(float(x_smooth[np.argmax(y_smooth)]), 1)
    opt_roi      = round(float(y_smooth.max()), 4)
    optimal_rows.append({'Tier': tier, 'Optimal_Discount_ROI': opt_discount,
                          'Predicted_Peak_ROI': opt_roi, 'Poly_Degree': DEGREE, 'R2': round(r2, 4)})
    ax.axvline(opt_discount, color=color, linestyle=':', linewidth=1.2, alpha=0.7)

ax.set_xlabel('Discount Level (%)', fontsize=12)
ax.set_ylabel('ROI', fontsize=12)
ax.set_title(f'Polynomial Regression (deg={DEGREE}) — Discount Level vs ROI by Tier (RQ6)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
save_figure(fig, 'rq6_discount_roi_polynomial.pdf')
plt.show()

Saved figure: /kaggle/working/rq6_discount_roi_polynomial.pdf


## 4. Satisfaction vs Discount Level by Tier

In [94]:
fig, ax = plt.subplots(figsize=(11, 6))

for tier, color in TIER_PALETTE.items():
    sub = df[df[TIER_COL] == tier].dropna(subset=['Discount_Level', 'Customer_Satisfaction_Post_Refund'])
    if len(sub) < 10:
        continue
    x = sub['Discount_Level'].values
    y = sub['Customer_Satisfaction_Post_Refund'].values

    ax.scatter(x, y, alpha=0.15, s=15, color=color, edgecolors='none')

    poly_pipe = fit_poly(x, y, degree=DEGREE)
    y_smooth  = poly_pipe.predict(x_smooth.reshape(-1, 1))
    r2        = r2_score(y, poly_pipe.predict(x.reshape(-1, 1)))

    ax.plot(x_smooth, y_smooth, color=color, linewidth=2.5,
            label=f'{tier} (R²={r2:.3f})')

    # Record optimal discount for satisfaction
    opt_sat_discount = round(float(x_smooth[np.argmax(y_smooth)]), 1)
    for row in optimal_rows:
        if row['Tier'] == tier:
            row['Optimal_Discount_Satisfaction'] = opt_sat_discount
            row['Predicted_Peak_Satisfaction']   = round(float(y_smooth.max()), 4)

ax.set_xlabel('Discount Level (%)', fontsize=12)
ax.set_ylabel('Customer Satisfaction (1–5)', fontsize=12)
ax.set_title(f'Discount Level vs Customer Satisfaction by Tier (RQ6)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
save_figure(fig, 'rq6_discount_satisfaction_scatter.pdf')
plt.show()

Saved figure: /kaggle/working/rq6_discount_satisfaction_scatter.pdf


## 5. Optimal Discount Heatmap

In [95]:
optimal_df = pd.DataFrame(optimal_rows)
save_table(optimal_df, 'rq6_optimal_discount_ranges.csv')

# Heatmap: Discount_Bin × Tier → Mean_ROI
tier_bin_pivot = df.groupby(['Discount_Bin', TIER_COL], observed=True)['ROI'].mean().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(tier_bin_pivot, annot=True, fmt='.3f', cmap='YlOrRd',
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Mean ROI'})
ax.set_title('Mean ROI by Discount Range and Subscription Tier (RQ6)', fontsize=13, fontweight='bold')
ax.set_xlabel('Subscription Tier')
ax.set_ylabel('Discount Range')
plt.tight_layout()
save_figure(fig, 'rq6_optimal_discount_by_tier.pdf')
plt.show()

print('Optimal Discount Ranges:')
print(optimal_df.to_string(index=False))

Saved table:  /kaggle/working/rq6_optimal_discount_ranges.csv
Saved figure: /kaggle/working/rq6_optimal_discount_by_tier.pdf
Optimal Discount Ranges:
    Tier  Optimal_Discount_ROI  Predicted_Peak_ROI  Poly_Degree     R2  Optimal_Discount_Satisfaction  Predicted_Peak_Satisfaction
   Basic                  69.0              2.8728            3 0.0007                           46.5                       2.5526
Standard                  35.3              2.7812            3 0.0008                           10.0                       2.5662
 Premium                  10.0              2.8251            3 0.0020                           10.0                       2.5728


## 6. Conclusions

In [96]:
print('=' * 60)
print('RQ6 CONCLUSION')
print('=' * 60)
print(f'Overall Pearson r (Discount vs ROI):          {r_roi:.4f}')
print(f'Overall Pearson r (Discount vs Satisfaction): {r_sat:.4f}')
print()
print('Optimal discount ranges by subscription tier (polynomial peak):')
for _, row in optimal_df.iterrows():
    print(f"  {row['Tier']:10s}: ROI peak at {row['Optimal_Discount_ROI']}%  "
          f"Satisfaction peak at {row.get('Optimal_Discount_Satisfaction', 'N/A')}%")
print()
print('Outputs saved:')
for f in ['rq6_discount_roi_polynomial.pdf','rq6_discount_satisfaction_scatter.pdf',
          'rq6_optimal_discount_by_tier.pdf',
          'rq6_discount_performance_summary.csv','rq6_optimal_discount_ranges.csv']:
    print(f'  {f}')

RQ6 CONCLUSION
Overall Pearson r (Discount vs ROI):          0.0099
Overall Pearson r (Discount vs Satisfaction): -0.0141

Optimal discount ranges by subscription tier (polynomial peak):
  Basic     : ROI peak at 69.0%  Satisfaction peak at 46.5%
  Standard  : ROI peak at 35.3%  Satisfaction peak at 10.0%
  Premium   : ROI peak at 10.0%  Satisfaction peak at 10.0%

Outputs saved:
  rq6_discount_roi_polynomial.pdf
  rq6_discount_satisfaction_scatter.pdf
  rq6_optimal_discount_by_tier.pdf
  rq6_discount_performance_summary.csv
  rq6_optimal_discount_ranges.csv
